# 01 — Marco Teórico: T5 + GEC + Transfer Learning

> **Objetivo.** Entender *por qué* T5 es la elección correcta para Corrección de Errores Gramaticales (GEC), antes de tocar una sola línea de código.

Este notebook es **puramente conceptual**. No requiere GPU ni dependencias — solo Markdown.

---

## 1. ¿Qué es T5?

**T5** (*Text-to-Text Transfer Transformer*, Raffel et al. 2020) es una familia de modelos **encoder-decoder** que reformula **toda tarea de NLP** como una conversión texto → texto.

```
clasificación  →  "sst2 sentence: I love this"      → "positive"
traducción     →  "translate English to German: ..." → "..."
resumen        →  "summarize: ..."                   → "..."
GEC (nuestra)  →  "grammar: I has a apple."          → "I have an apple."
```

La unificación texto-a-texto es lo que hace que T5 sea conceptualmente elegante: un mismo modelo, una misma loss (cross-entropy sobre tokens), distintos prefijos.

### Arquitectura — Transformer encoder-decoder

```
entrada "grammar: I has a apple."
   │
   ▼ tokenizer (SentencePiece)
[ids]
   │
   ▼  Encoder (N capas self-attention + FFN)
[hidden_states]
   │
   ▼  Decoder (N capas masked-self-attention + cross-attention + FFN)
[logits sobre vocab]
   │
   ▼  arg-max / beam search
salida "I have an apple."
```

### Pre-entrenamiento — Span corruption

T5 fue pre-entrenado en **C4** (~750 GB de inglés filtrado de Common Crawl) con el objetivo de **span corruption**:

```
original:  "Thank you for inviting me to your party last week."
corrupto:  "Thank you <X> me to your party <Y> week."
target:    "<X> for inviting <Y> last"
```

El modelo aprende a **regenerar tramos enmascarados** del texto. Esto es funcionalmente equivalente a *"reescribir un fragmento de texto roto"* — exactamente lo que hace GEC.

---

## 2. ¿Qué es GEC?

**Grammatical Error Correction** = dada una oración con errores (gramática, ortografía, concordancia, preposiciones, artículos), producir una versión corregida que preserve el significado.

### Tipos de error que se atacan

| Categoría | Ejemplo erróneo | Corrección |
|---|---|---|
| Concordancia sujeto-verbo | He **go** to school | He **goes** to school |
| Tiempo verbal | I **eat** pizza yesterday | I **ate** pizza yesterday |
| Artículos | I have **a apple** | I have **an apple** |
| Preposiciones | Depends **of** | Depends **on** |
| Plural / singular | Many **child** | Many **children** |
| Ortografía | recieve | receive |

### Por qué GEC es difícil

1. **Múltiples correcciones válidas** — la oración objetivo no es única.
2. **Cambios mínimos** — la métrica BLEU clásica sobre-penaliza ediciones pequeñas.
3. **Sensibilidad al contexto** — "I read a book" es válido en pasado o presente; depende del contexto.
4. **Cobertura de errores raros** — el modelo debe generalizar a errores que no vio en entrenamiento.

---

## 3. Transfer Learning vs Fine-Tuning vs PEFT

```
┌────────────────────────────────────────────────────────────────┐
│  TRANSFER LEARNING (concepto general)                          │
│                                                                │
│  modelo pre-entrenado (C4, 750GB) ──► tarea downstream (GEC)   │
└────────────────────────────────────────────────────────────────┘
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
┌───────────────┐ ┌──────────┐ ┌─────────────┐
│ Feature       │ │  FULL    │ │ PEFT (LoRA) │
│ extraction    │ │  FINE-   │ │             │
│ (freeze base) │ │  TUNING  │ │ entrena <1% │
└───────────────┘ └──────────┘ └─────────────┘
  rápido            calidad      balance
  poca calidad      alto costo   buen calidad/costo
```

| Estrategia | Qué se entrena | Memoria | Calidad | Cuándo usar |
|---|---|---|---|---|
| Feature extraction | Cabeza nueva, base congelada | Baja | Baja | Tareas muy similares al pre-entreno |
| **Full fine-tuning** | **Todos los parámetros** | **Alta** | **Alta** | **Datos suficientes y compute disponible** ← *PoC notebook 05* |
| PEFT / LoRA | Adaptadores de bajo rango (~0.5%) | Baja | Cercana a full | Modelos grandes en hardware modesto ← *PoC notebook 08* |
| Prompt tuning | Solo embeddings de prompt | Mínima | Media | Muchas tareas con un solo modelo |

---

## 4. Por qué T5 sobre alternativas

| Modelo | Familia | Pros para GEC | Contras |
|---|---|---|---|
| **T5** | Encoder-decoder | Pre-entreno = denoising natural; prefijos de tarea explícitos | — |
| BART | Encoder-decoder | Similar a T5, también denoising | Tokenizer sin prefijos de tarea estandarizados |
| GPT-2/3/4 | Decoder-only | Generación fluida | Requiere prompt engineering pesado, caro de fine-tunear |
| BERT | Encoder-only | Bueno para clasificación | **No genera texto** → inadecuado para GEC |
| LLaMA / Mistral | Decoder-only LLM | SOTA en cero-shot | Overkill para GEC, costo elevado |

**Conclusión.** T5 es el sweet spot: pre-entreno alineado con la tarea, arquitectura encoder-decoder ideal para seq2seq, tamaños desde 60M (small) hasta 11B (xxl) con la misma API.

---

## 5. Formato de la tarea — el prefijo `grammar:`

T5 distingue tareas por **prefijo de texto**, no por embeddings especiales:

```
entrada modelo  ←  "grammar: " + oracion_original
salida esperada ←  oracion_corregida
```

El prefijo es **arbitrario pero debe ser consistente** entre training e inferencia. La PoC usa `"grammar: "` siguiendo la convención de la guía.

---

## 6. Hardware — por qué Apple Silicon es viable

- **Memoria unificada** — la GPU comparte RAM con la CPU. Un Mac de 16 GB puede cargar modelos que en una GPU discreta de 8 GB no caben.
- **MPS** (*Metal Performance Shaders*) — backend de PyTorch que ejecuta tensor ops en la GPU integrada del M1/M2/M3.
- **bf16 nativo** — los chips Apple Silicon soportan bfloat16 → mismo rango dinámico que fp32, mitad de memoria.

Limitaciones:
- No DDP / multi-GPU.
- Algunos operadores caen a CPU (gestionable con `PYTORCH_ENABLE_MPS_FALLBACK=1`).
- fp16 menos estable que en CUDA → preferir bf16.

---

## 7. Plan de la PoC

```
01 (este)   Teoría
02          Verificar MPS + dependencias
03          EDA del dataset coedit
04          Tokenización + DataCollator
05          Fine-tuning con Seq2SeqTrainer
06          Métricas: SacreBLEU + GLEU + Exact-Match
07          Inferencia + score de confianza por logits
08 (opc.)   Escalar a t5-base con LoRA
```

**Siguiente notebook → `02_setup_entorno_mac.ipynb`**

---

## Referencias

- Raffel et al. (2020) — *Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer*. https://arxiv.org/abs/1910.10683
- Bryant et al. (2017) — *Automatic Annotation and Evaluation of Error Types for Grammatical Error Correction* (ERRANT).
- Raheja et al. (2023) — *CoEdIT: Text Editing by Task-Specific Instruction Tuning*. https://arxiv.org/abs/2305.09857
- Hu et al. (2021) — *LoRA: Low-Rank Adaptation of Large Language Models*. https://arxiv.org/abs/2106.09685
